# SPAM MAIL DETECTION

In [65]:
import pandas as pd
import re
import nltk
import pickle as pk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [66]:
mail_data=pd.read_csv('spam.csv',encoding='latin-1')

In [67]:
mail_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [68]:
mail_data.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [69]:
mail_data.shape

(5572, 5)

In [70]:
for column in mail_data.columns:
    print(f"Column: {column}")
    print(len(mail_data[column].unique()))


Column: v1
2
Column: v2
5169
Column: Unnamed: 2
44
Column: Unnamed: 3
11
Column: Unnamed: 4
6


In [71]:
new_mail_data = mail_data.where((pd.notnull(mail_data)), '')

In [72]:
new_mail_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",,,
1,ham,Ok lar... Joking wif u oni...,,,
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,,,
3,ham,U dun say so early hor... U c already then say...,,,
4,ham,"Nah I don't think he goes to usf, he lives aro...",,,


### Label Encoding

In [73]:
new_mail_data.loc[new_mail_data['v1'] == 'spam', 'v1'] = 0
new_mail_data.loc[new_mail_data['v1'] == 'ham', 'v1'] = 1

In [74]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [75]:
ps = PorterStemmer()

In [76]:
stop_words = set(stopwords.words('english'))

In [77]:
print(stopwords.words('english')[:10])

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


In [78]:
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(r'http\S+', ' ', text)   
    text = re.sub(r'[^a-zA-Z]', ' ', text)   
    words = text.split()    
    words = [word for word in words if word not in stop_words]    
    words = [ps.stem(word) for word in words]    
    return " ".join(words)

In [79]:
new_mail_data['processed_text'] = new_mail_data['v2'].apply(preprocess_text)

spam - 0<br>
ham - 1

In [80]:
X = new_mail_data['processed_text']
Y = new_mail_data['v1']

In [81]:
print(X)

0       go jurong point crazi avail bugi n great world...
1                                   ok lar joke wif u oni
2       free entri wkli comp win fa cup final tkt st m...
3                     u dun say earli hor u c alreadi say
4                    nah think goe usf live around though
                              ...                        
5567    nd time tri contact u u pound prize claim easi...
5568                                b go esplanad fr home
5569                                    piti mood suggest
5570    guy bitch act like interest buy someth els nex...
5571                                       rofl true name
Name: processed_text, Length: 5572, dtype: object


In [82]:
print(Y)

0       1
1       1
2       0
3       1
4       1
       ..
5567    0
5568    1
5569    1
5570    1
5571    1
Name: v1, Length: 5572, dtype: object


In [83]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3)

In [84]:
print(X.shape)
print(X_train.shape)
print(X_test.shape)

(5572,)
(4457,)
(1115,)


In [85]:
vectorizer = TfidfVectorizer()
X_train_features = vectorizer.fit_transform(X_train)
X_test_features = vectorizer.transform(X_test)


In [86]:
print(X_train)

3075    mum hope great day hope text meet well full li...
1787                                   ye sura sun tv lol
1614                     sef dey laugh meanwhil darl anji
4304                                   yo come carlo soon
3266                               ok come n pick u engin
                              ...                        
789                            gud mrng dear hav nice day
968                                 will go aptitud class
1667           dad gonna call get work ask crazi question
3321    ok darlin supos ok worri much film stuff mate ...
1688                           nan sonathaya soladha boss
Name: processed_text, Length: 4457, dtype: object


In [87]:
print(X_train_features)

  (0, 3125)	0.29473823593356974
  (0, 2198)	0.43070194535584805
  (0, 1993)	0.44582374908675926
  (0, 1141)	0.3688325805843802
  (0, 4834)	0.19671033581522265
  (0, 2935)	0.21758099038206752
  (0, 5346)	0.22139463068022516
  (0, 1826)	0.30299014669442126
  (0, 2697)	0.24418433444341897
  (0, 11)	0.32611021837325754
  (1, 5544)	0.35056971070320353
  (1, 4701)	0.5652509076654626
  (1, 4678)	0.4769136859540388
  (1, 5061)	0.4306015894277422
  (1, 2752)	0.380431198316959
  (2, 4196)	0.45329962489137066
  (2, 1222)	0.3811461667946165
  (2, 2645)	0.34506943774623944
  (2, 2927)	0.41722289584299355
  (2, 1131)	0.3880961710740478
  (2, 186)	0.45329962489137066
  (3, 5567)	0.524841815062256
  (3, 923)	0.34970933876863236
  (3, 731)	0.5979251862237673
  (3, 4466)	0.49470184881344015
  :	:
  (4453, 5399)	0.5590229330608766
  (4453, 241)	0.6529435350028746
  (4454, 1895)	0.2411415100515692
  (4454, 3849)	0.4037465537667388
  (4454, 5460)	0.32186686620781463
  (4454, 691)	0.2170136648551823
  (4454

In [88]:
Y_train = Y_train.astype('int')
Y_test = Y_test.astype('int')

In [89]:
smote = SMOTE(random_state=42)

In [90]:
X_train_smote, Y_train_smote = smote.fit_resample(X_train_features, Y_train)

In [91]:
print(Y_train_smote.value_counts())

v1
1    3865
0    3865
Name: count, dtype: int64


In [92]:
models = {
    "Logistic Regression": LogisticRegression(),
    "SVM": LinearSVC(),
}

In [93]:
for name, model in models.items():

    print("-"*60)
    print(name)
    print("-"*60)

    model.fit(X_train_smote, Y_train_smote)

    train_pred = model.predict(X_train_features)

    print("\nTRAINING RESULTS")
    print("Accuracy:", accuracy_score(Y_train, train_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_train, train_pred))
    print("Classification Report:\n", classification_report(Y_train, train_pred))

    test_pred = model.predict(X_test_features)

    print("\nTEST RESULTS")
    print("Accuracy:", accuracy_score(Y_test, test_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_test, test_pred))
    print("Classification Report:\n", classification_report(Y_test, test_pred))

    print("\n\n")

------------------------------------------------------------
Logistic Regression
------------------------------------------------------------

TRAINING RESULTS
Accuracy: 0.9885573255553063
Confusion Matrix:
 [[ 568   24]
 [  27 3838]]
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.96      0.96       592
           1       0.99      0.99      0.99      3865

    accuracy                           0.99      4457
   macro avg       0.97      0.98      0.98      4457
weighted avg       0.99      0.99      0.99      4457


TEST RESULTS
Accuracy: 0.9802690582959641
Confusion Matrix:
 [[140  15]
 [  7 953]]
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.90      0.93       155
           1       0.98      0.99      0.99       960

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98

In [94]:
with open("vectorizer.pkl", "wb") as file:
    pk.dump(vectorizer, file)

In [95]:
with open("logistic_regression_model.pkl", "wb") as file:
    pk.dump(models["Logistic Regression"], file)

In [96]:
with open("svm_model.pkl", "wb") as file:
    pk.dump(models["SVM"], file)

In [97]:
import pickle as pk

# Load trained model
with open("model.pkl", "rb") as file:
    loaded_model = pk.load(file)

# Load TF-IDF vectorizer
with open("vectorizer.pkl", "rb") as file:
    loaded_vectorizer = pk.load(file)

# New email to test
input_mail = ["Congratulations! You have won a free iPhone. Click here to claim your prize."]

# Convert text to TF-IDF features
input_data_features = loaded_vectorizer.transform(input_mail)

# Predict
prediction = loaded_model.predict(input_data_features)

if prediction[0] == 0:
    print("Spam Mail")
else:
    print("Ham Mail")

ValueError: X has 5606 features, but LinearSVC is expecting 5480 features as input.